# CCA Developer · M1 — MSO Foundations

This lab covers the basics you need before building anything with Claude: what tokens are, how the context window is shared, why answers vary, which model to pick, when to turn on thinking, how to prompt, and how to call Claude in production. It uses the AWS Customer Agreement and a contract-review assistant as the working example.

| Section | What you learn | Subtopics |
|---|---|---|
| **1.1 Tokens** | See what Claude actually reads, and what it costs | What a token is<br>Counting tokens before you send |
| **1.2 Context Window** | Budget input and output together, and order a prompt so it can be cached | `input + max_tokens` rule<br>Fill order and prompt caching |
| **1.3 Sampling and Non-Determinism** | Understand why the same prompt gives different answers, and test for it | Temperature and other parameters<br>Designing around non-determinism |
| **1.4 The Claude Model Family** | Choose a model with evidence, not intuition | Haiku, Sonnet, Opus, Fable<br>Alias vs pinned IDs |
| **1.5 Reasoning Modes** | Know when extended thinking is worth paying for | Basic mode<br>Extended thinking |
| **1.6 Prompting Modes** | Add only as much guidance as the task needs | Zero-shot<br>Few-shot<br>Multi-turn |
| **1.7 SDK vs REST vs Batch** | Pick the right way to call Claude for each job | SDK vs raw REST<br>Synchronous, streaming, batch |

### Before you start
- Have your **Anthropic API key** ready.
- Run the cells from top to bottom. Every cell that calls Claude uses a small number of tokens. Sections 1.2, 1.4 and 1.7 send the whole contract, so they use the most.

> ⚠️ This notebook runs on a throw-away Colab machine. Your API key is never saved in the notebook.

## Setup

Install the tools used in this lab.

| Package | Used for |
|---|---|
| `anthropic` | Calling Claude with the official SDK |
| `PyMuPDF` | Reading the contract PDF |
| `requests` | Showing the raw REST call in 1.7 (already in Colab) |

**Install:** install the tools we need.

In [ ]:
# Install the Python libraries we need
!pip install -q -U anthropic PyMuPDF

Now let's set things up. This cell does three things:
- **Connects to Claude** using your API key. You paste it when asked, and it is not saved.
- **Loads the contract.** You upload the PDF, and it is split into pages, so you can look at one page or at the whole contract.
- **Adds `ask()`**, a small helper that sends one prompt to Claude and returns the answer text.

Every later step reuses these.

📥 **Download the contract first:** [AWS1.pdf](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing). Save it to your computer, then select it when the upload box appears.

**Setup:** ask for your API key, upload the contract PDF and cut it into pages. It also defines `ask()`, which sends one prompt to Claude and returns the answer.

In [ ]:
import os, re, json, time
from getpass import getpass
import anthropic, fitz
from google.colab import files

os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY

SONNET = "claude-sonnet-5-5"   # our default model
HAIKU = "claude-haiku-4-5"     # fast and cheap
OPUS = "claude-opus-5-5"       # most capable (only named in 1.4, never called)

# Upload AWS1.pdf and split it into pages
name = next(iter(files.upload()))
with fitz.open(name) as pdf:
    PAGES = [p.get_text() for p in pdf]
contract = "\n\n".join(PAGES)


def ask(prompt, model=HAIKU, max_tokens=500, system=None, **options):
    """Send one prompt and return Claude's text answer."""
    extra = {"system": system} if system else {}
    r = client.messages.create(model=model, max_tokens=max_tokens,
                               messages=[{"role": "user", "content": prompt}], **extra, **options)
    return "".join(b.text for b in r.content if b.type == "text").strip()


print(len(PAGES), "pages |", f"{len(contract):,}", "characters")

## 1.1 Tokens

Everything Claude processes is **tokens**, not characters or words. Tokens are the foundation for reasoning about cost, speed and context limits.

Claude uses a **BPE (Byte Pair Encoding)** tokeniser. Common English words are usually one token. Uncommon words, code symbols and non-Latin scripts are split across several.

| Content type | Approximate token count |
|---|---|
| 1 average English word | ~1.3 tokens |
| 1 line of Python | ~5–12 tokens |
| 1 character in Chinese, Japanese or Korean | ~2–3 tokens |
| 1 JSON object with 5 fields | ~20–40 tokens |
| 1 image (any size) | 1,000–3,000 tokens (detail-dependent) |

> **Note:** structured output costs more than prose. A JSON envelope around an answer spends tokens on `{`, `"`, `:` and `}` at every level of nesting. This matters at scale.

### Counting tokens before you send

Use the SDK's token counter before an expensive request. It is free to call.

Now let's count. We write the same termination fact three ways, as a sentence, as compact JSON and as indented JSON, and ask Claude's counter how many tokens each one costs. Then we count one full page of the contract to see the characters-per-token ratio on real text.

**Token count:** count the same fact as prose, compact JSON and indented JSON. Then count page 1 of the contract.

In [ ]:
def count(text, model=SONNET):
    """Input tokens for a prompt (includes a few tokens of message overhead, so compare numbers relatively)."""
    return client.messages.count_tokens(
        model=model, messages=[{"role": "user", "content": text}]
    ).input_tokens


fact = {"termination": {"by": ["customer", "provider"], "notice_days": 30, "form": "written"}}
samples = {
    "Prose":          "Either party may terminate this agreement with thirty days written notice.",
    "Compact JSON":   json.dumps(fact),
    "Indented JSON":  json.dumps(fact, indent=4),
}
for label, text in samples.items():
    print(f"{label:<14} {len(text):>3} characters -> {count(text):>3} tokens")

page = PAGES[0]
n = count(page)
print(f"\nPage 1 of the contract: {len(page):,} characters -> {n:,} tokens (about {len(page) / n:.1f} characters per token)")

**Result:** the same information costs more as JSON than as a sentence, and indenting costs more again, because every brace, quote and space is a token. On normal English text, one token is roughly 4 characters. Tokens, not characters or pages, are what you pay for and what limits a request.

## 1.2 Context Window

The **context window** is the total number of tokens Claude can hold in one conversation, input and output combined. It is a **shared resource**: the contract, your instructions **and** the answer all have to fit.

When you call the API you set `max_tokens`, the most output the model may generate. The input (system prompt plus conversation history) and the output must fit in the window together:

```text
input_tokens + max_tokens ≤ context_window_size
```

If your input is 180K tokens and you request `max_tokens=32768`, you need a window of at least 212,768 tokens.

### The budget rule

Now let's apply the rule. We count the real contract, then check three cases against a 200K window: the contract as it is, the guide's 180K example, and a tiny input that asks for a huge answer.

**Budget check:** count the contract, then test `input + max_tokens` against a 200K window for three cases.

In [ ]:
CONTEXT_WINDOW = 200_000   # tokens, for illustration. Check the docs for the window of the model you use.
contract_tokens = count(contract)

cases = [
    ("Our contract, 1,000 token answer",   contract_tokens, 1_000),
    ("180K input, 32,768 token answer",    180_000,         32_768),
    ("Short question, 250K token answer",  50,              250_000),
]
for label, n_input, max_tokens in cases:
    needed = n_input + max_tokens
    verdict = "✅ fits" if needed <= CONTEXT_WINDOW else "❌ too big"
    print(f"{label:<36} {n_input:>7,} + {max_tokens:>7,} = {needed:>8,} of {CONTEXT_WINDOW:,} -> {verdict}")

**Result:** the contract is a small share of the window, so it fits easily. The second case does not fit, even though the input alone does, because the output budget counts too. Always budget both.

### Fill order and prompt caching

The model keeps a **Key-Value (KV) cache** of the token representations it has already computed, so tokens that were already processed do not need to be computed again. The cache is bounded by the context window. When a conversation grows past the limit, you must decide what to drop.

Because of the cache, **the order of your prompt matters**. Put stable content first and dynamic content last:

1. **System prompt**: stable, cached first
2. **Static documents or reference material**: for example the contract
3. **Conversation history**: older turns first
4. **Current user message**: dynamic, at the very end

Stable content at the front gets cached and reused. Dynamic content at the end does not invalidate it.

> ⚠️ Do not put dynamic content (timestamps, user IDs) at the front of your system prompt. It changes the start of the prompt on every request, invalidates the cache on every request and removes the cost savings of prompt caching.

Now let's see both orders. We ask two questions about the contract each way and print the cache numbers from the response. `cache_control` marks the end of the part to cache.
- **Good order:** instructions, then the contract, then the question. The first call writes the cache and the second one reads it.
- **Bad order:** a timestamp is placed first. The start of the prompt differs on every call, so the second call cannot read the cache.

**Cache order:** run two questions with the stable-first order and two with a timestamp at the front, and print the cache usage of each call.

In [ ]:
INSTRUCTIONS = "You are a contract analyst. Answer only from the contract, in one sentence."


def ask_cached(question, stamp=None):
    """One question about the contract. `stamp` puts a dynamic value at the FRONT of the system prompt."""
    system = [{"type": "text", "text": INSTRUCTIONS},                        # 1) stable instructions
              {"type": "text", "text": f"Contract:\n{contract}",           # 2) stable document
               "cache_control": {"type": "ephemeral"}}]                      #    cache everything up to here
    if stamp:
        system.insert(0, {"type": "text", "text": f"Current time: {stamp}"})   # dynamic content first: the mistake
    r = client.messages.create(model=SONNET, max_tokens=150, system=system,
                               messages=[{"role": "user", "content": question}])   # 3) dynamic question LAST
    u = r.usage
    print(f"  cache written: {u.cache_creation_input_tokens or 0:>6,} | cache read: {u.cache_read_input_tokens or 0:>6,} | uncached input: {u.input_tokens:>6,}")


print("Good order (stable first)")
ask_cached("Who are the parties?")
ask_cached("What is the governing law?")

print("Bad order (timestamp first)")
ask_cached("Who are the parties?", stamp=time.time())
ask_cached("What is the governing law?", stamp=time.time())
# If every number is 0, the prompt is shorter than the model's minimum cacheable size.

**Result:** with the good order, the second call **reads** the contract from the cache instead of processing it again. With the timestamp first, the second call reads nothing and writes the whole cache again, because the front of the prompt changed. (If the first good call already shows a cache read, someone ran this cell in the last few minutes and the cache was still warm.)

## 1.3 Sampling and Non-Determinism

Claude generates text **one token at a time**. At each step it gives every possible next token a probability, then **samples** one. This is why the same prompt can produce different outputs on different runs. This is **non-determinism**, and it is fundamental to how the model works.

```text
Next token after "Notice period is ..."
"30"     60%     "thirty" 20%     "90" 10%     other 10%
```

### Sampling parameters

| Parameter | What it controls | Range | Practical use |
|---|---|---|---|
| `temperature` | Width of the probability distribution | 0–1 | Low (0–0.3) for factual or extraction tasks. High (0.7–1) for creative tasks |
| `top_p` | Nucleus sampling: cumulative probability cutoff | 0–1 | Use `temperature` **or** `top_p`, not both |
| `top_k` | Hard cap on vocabulary candidates | 1–N | Less commonly tuned. Anthropic recommends sticking with `temperature` |
| `max_tokens` | Maximum output length | 1–model max | Set it explicitly. Do not rely on the default |

Now let's see temperature at work. We ask the same question three times at `temperature=0` and three times at `temperature=1`, and print each answer.

**Temperature:** ask the same termination question 3 times at temperature 0 and 3 times at temperature 1.

In [ ]:
question = "In one sentence, describe how either party can terminate this contract.\n\n" + contract

for temp in (0, 1):            # the most consistent setting vs the most varied one
    print(f"--- temperature={temp} ---")
    for run in (1, 2, 3):      # same prompt, three runs: do the answers change?
        print(f"run {run}:", ask(question, max_tokens=150, temperature=temp), "\n")

**Result:** at temperature 1 the wording moves around from run to run. At temperature 0 the answers are the same or very close.

### Temperature is not the only source of variation

Even with `temperature=0`, outputs can vary because of:
- **Hardware:** floating-point differences across server instances
- **Model updates:** a new model version changes behaviour
- **Token budget pressure:** a model close to its `max_tokens` limit may abbreviate
- **Prompt caching:** subtle timing differences

> `temperature=0` gives **near**-deterministic output, not perfectly deterministic output. For repeatable behaviour, use `temperature=0` and **pin the model ID** (see 1.4). Do not rely on `top_p` or `top_k` to fix variation.

### Designing around non-determinism

Do not fight it. Design around it:
- Define expected **properties** of the output in your eval, not exact strings
- Use **structured output** (a JSON schema or tool use) to constrain the format
- Test across **multiple samples** to catch edge-case behaviour

Now let's compare the two kinds of test. We sample the same question 5 times at temperature 1. An **exact-match** test asks "is the answer the same sentence every time?". A **property** test asks "does the answer name a notice period as a number of days?".

**Test design:** collect 5 answers at temperature 1, then grade them by exact match and by a property check.

In [ ]:
q = ("How many days of notice must be given to terminate this contract for convenience? "
     "Reply in one sentence.\n\n" + contract)
answers = [ask(q, max_tokens=100, temperature=1) for _ in range(5)]

for a in answers:
    print("-", a)

exact = len(set(answers)) == 1                                    # same sentence every time?
prop = all(re.search(r"\b\d+\b.*days|days.*\b\d+\b", a, re.I) for a in answers)   # always names a number of days?
print("\nExact-match test passes: ", exact)
print("Property test passes:    ", prop)

**Result:** the wording differs, so the exact-match test fails even though every answer may be correct. The property test passes as long as each answer states the notice period. Test the **meaning**, not the exact string.

## 1.4 The Claude Model Family

There is no "best" model, only trade-offs between **capability, latency and cost**. The family has four tiers:

| Model | Use case | Strengths | When *not* to use |
|---|---|---|---|
| **Claude Haiku 4.5** | High-volume, latency-sensitive tasks | Fastest and cheapest. Good for extraction and classification | Complex reasoning, long-horizon tasks |
| **Claude Sonnet 5** | General purpose (the default) | Best balance of capability, speed and cost | When you have confirmed Sonnet fails your eval |
| **Claude Opus 5** | Complex research, long-horizon reasoning | Highest capability, extended thinking | High-volume use cases (cost-prohibitive) |
| **Claude Fable 5** | Narrative, roleplay, creative generation | Optimised for engaging long-form fiction | Factual Q&A, structured data extraction |

### Model selection rule

1. **Start with Sonnet.**
2. Move **up** to Opus only if Sonnet fails your eval benchmarks on your specific task.
3. Move **down** to Haiku only if Sonnet passes but your cost or latency budget is not met.

> ⚠️ *"This task sounds complex"* is not a valid reason to choose Opus over Sonnet. Run the eval on your actual task first. Sonnet handles most production workloads.

For a contract-review assistant that means: Haiku for high-volume extraction (parties, dates, amounts), Sonnet for clause analysis, and Opus only for something Sonnet fails on, such as long multi-document reasoning. Fable is for stories, so it has no place here.

### Alias vs pinned IDs

A model is chosen by an ID string. There are two kinds:
- **Alias**, such as `claude-sonnet-5`: always resolves to the latest stable release of that model. It updates automatically, so use it for **development**.
- **Pinned**, such as `claude-sonnet-5-20251101`: resolves to one specific release. Nothing changes under you. Use it whenever you have a validated eval suite. M5 covers the versioning strategy in detail.

Now let's see the IDs your key can use, then run the same extraction on Haiku and on Sonnet and compare the time and the output size.

**Models:** list the available model IDs. Then run one extraction on Haiku and on Sonnet and compare time and tokens.

In [ ]:
# Dated IDs are pinned releases. IDs without a date are aliases.
for m in list(client.models.list(limit=20))[:12]:
    print(f"{m.id:<40} {m.display_name}")

print()
task = ("Extract the parties, the notice period for termination and the governing law. "
        "Answer in 3 short bullets.\n\n" + contract)

for label, model in [("Haiku", HAIKU), ("Sonnet", SONNET)]:
    start = time.time()
    r = client.messages.create(model=model, max_tokens=300, messages=[{"role": "user", "content": task}])
    print(f"--- {label} · {time.time() - start:.1f}s · {r.usage.input_tokens:,} input / {r.usage.output_tokens} output tokens ---")
    print(r.content[0].text, "\n")

**Result:** both models can extract these simple facts, and Haiku does it faster and cheaper. That is what a Haiku-to-Sonnet decision looks like: if Haiku passes your eval on the real task, use it. If it does not, use Sonnet. The answer comes from your eval, not from how hard the task sounds.

## 1.5 Reasoning Modes

Claude has two reasoning modes:
- **Basic (default):** a direct response. Reasoning happens implicitly while the answer is generated. Fast, and enough for the vast majority of tasks.
- **Extended thinking:** Claude first writes a scratchpad of reasoning tokens (a visible **thinking block**), then the final answer. This significantly improves accuracy on multi-step math and logic, complex code review that holds many constraints at once, and long-horizon planning.

| Task type | Extended thinking needed? |
|---|---|
| Simple Q&A, summarisation, extraction | No |
| Code generation for standard patterns | No |
| Multi-step mathematical proof | Yes |
| Complex architectural trade-off analysis | Usually yes |
| Hard competitive programming problems | Yes |

You turn it on with the `thinking` parameter:

```python
thinking={"type": "enabled", "budget_tokens": 10000}   # max tokens for the thinking block
```

> **Cost:** thinking tokens count as **output tokens**. `budget_tokens` is a *ceiling*, not a target: the model uses what it needs, up to the limit. Only enable thinking once basic mode has failed your eval. `max_tokens` must be larger than `budget_tokens`, because thinking and the answer share it.

Now let's compare. We ask a contract risk question in basic mode and with extended thinking, and print the block types, the output tokens and the answer. In the second run we also print the thinking itself.

**Thinking:** ask one risk question in basic mode and with a 2,000-token thinking budget, and compare the output tokens.

In [ ]:
risk_question = ("Is the termination clause risky for the customer? Answer in 3 sentences.\n\n" + contract)

modes = [
    ("Basic mode", None),
    ("Extended thinking (budget 2000)", {"type": "enabled", "budget_tokens": 2000}),
]
for label, thinking in modes:
    options = {"thinking": thinking} if thinking else {}
    r = client.messages.create(model=SONNET, max_tokens=4000,
                               messages=[{"role": "user", "content": risk_question}], **options)
    print(f"--- {label} ---")
    print("blocks:", [b.type for b in r.content], "| output tokens:", r.usage.output_tokens)
    for block in r.content:
        if block.type == "thinking":
            print("Thinking:", block.thinking[:600], "...\n")
        elif block.type == "text":
            print("Answer:", block.text, "\n")

**Result:** the extended-thinking response has an extra `thinking` block, and its output token count is higher. Those extra tokens are the cost of the scratchpad. For a three-sentence summary like this one, basic mode is usually enough, which is exactly the point: run the eval first, then pay for thinking only if basic mode fails.

## 1.6 Prompting Modes

Three prompting modes form a progression of specificity:

| Mode | Examples | Use when |
|---|:-:|---|
| **Zero-shot** | 0 | The task is clearly specified and the model's training already covers the pattern |
| **Few-shot** (one-shot or multi-shot) | 1 or more | The output format matters and words alone do not specify it, or the task needs a style or reasoning pattern the model has to see |
| **Multi-turn** | History | Interactive assistants. Be deliberate about how much history you keep |

Guidelines for few-shot examples:
- **Quality over quantity:** three excellent, diverse examples beat ten mediocre ones
- **Cover the hard cases:** easy examples add no information. Edge cases teach the boundary
- **Match the format exactly:** if production input has typos, include a typo in an example
- **Keep examples adjacent** to the instruction they illustrate

### Zero-shot and few-shot

Now let's label a contract clause. The clause below mentions "30 days" but is about **payment**, not termination, so it is a good test. We ask two ways:
- **Zero-shot:** instructions only.
- **Few-shot:** the same instructions plus three examples, one of which has a typo and one of which has "30 days" in a termination clause, so the model sees the hard cases.

**Zero vs few-shot:** label one payment clause with no examples, then with three examples, and compare the format of the answers.

In [ ]:
clause = "Fees are due within 30 days of the invoice date. Overdue amounts accrue interest at 1.5% per month."

zero_shot = f"""
Classify this contract clause as payment, termination, liability or other.

Clause: {clause}
"""

few_shot = f"""
Classify the contract clause as payment, termination, liability or other.
Reply with one word only.

Clause: "Either party may end this agreement with 30 days' notice."
Label: termination

Clause: "We r not liable for lost profits or indirect damagez."
Label: liability

Clause: "Invoices are payable net 30. Taxes are the customer's responsibility."
Label: payment

Clause: "{clause}"
Label:
"""

print("Zero-shot:", ask(zero_shot, max_tokens=200), "\n")
print("Few-shot: ", ask(few_shot, max_tokens=20))

**Result:** zero-shot usually gets the right label but wraps it in an explanation, so the format is up to the model. Few-shot returns one word in the exact format we showed. The typo in an example is deliberate: real clauses are messy, so the examples should be too.

> More examples is not always better. Ask: **"What is the smallest amount of guidance that gives the quality we need?"**

### Multi-turn

The Messages API is **stateless**: Claude only knows what is in the `messages` array you send. For a multi-turn conversation, you pass the whole prior exchange each time.

Now let's hold a short conversation. The second question, "And which law governs it?", only makes sense with the first turn, so we append Claude's reply and the follow-up to the history before the second call.

**Multi-turn:** ask who the parties are, add the reply to the history, then ask a follow-up that depends on it.

In [ ]:
# Turn 1: a question about the contract
history = [{"role": "user", "content": f"Contract:\n{contract}\n\nWho are the parties?"}]
first = client.messages.create(model=HAIKU, max_tokens=300, messages=history)
print("Turn 1:", first.content[0].text, "\n")

# Keep Claude's reply in the history, then add a follow-up that only makes sense with the earlier turns
history.append({"role": "assistant", "content": first.content[0].text})
history.append({"role": "user", "content": "And which law governs it?"})
second = client.messages.create(model=HAIKU, max_tokens=300, messages=history)
print("Turn 2:", second.content[0].text, "\n")

print("Messages sent on the 2nd call:", len(history), "| input tokens:", f"{second.usage.input_tokens:,}")

**Result:** Claude understood "it" because the earlier turns were in the history. The history grows with every turn, and so does the number of input tokens you pay for, so decide deliberately how much history to keep.

## 1.7 SDK vs REST vs Batch

### SDK vs raw REST

Every call is an HTTPS request to `/v1/messages`. The official **Python and TypeScript SDKs** wrap it and add:
- **Automatic retries** with exponential backoff, so rate-limit errors are handled for you
- **Streaming parsing**, event by event, without manual SSE handling
- **Type-safe** request and response objects
- **Helper utilities**, such as token counting and tool schema validation

Use the SDK for all production integrations. Use raw REST only when the SDK is unavailable, such as an unusual runtime or an edge function with size limits.

Now let's make the same request both ways: with `requests` (you build the headers and parse the JSON) and with the SDK.

**SDK vs REST:** ask who the parties are from page 1, once with raw REST and once with the SDK.

In [ ]:
import requests

payload = {
    "model": HAIKU,
    "max_tokens": 150,
    "messages": [{"role": "user", "content": "Who are the parties to this agreement? One sentence.\n\n" + PAGES[0]}],
}

# Raw REST: you build the headers, parse the JSON, and handle errors and retries yourself
raw = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"],
             "anthropic-version": "2023-06-01",
             "content-type": "application/json"},
    json=payload, timeout=60)
raw.raise_for_status()
print("Raw REST:", raw.json()["content"][0]["text"], "\n")

# SDK: the same call with typed objects. max_retries adds backoff on rate-limit and server errors
sdk = anthropic.Anthropic(max_retries=5)
r = sdk.messages.create(**payload)
print("SDK:     ", r.content[0].text)

**Result:** the answers match. The SDK call is shorter, gives you objects instead of raw JSON, and retries for you.

### Synchronous vs streaming vs batch

Choose by the user experience:

| Mode | When to use | Latency UX |
|---|---|---|
| **Synchronous** | Internal pipelines, short outputs, programmatic processing | Full wait. Not suitable for interactive UIs |
| **Streaming** | Any user-facing output | First token in about 500 ms. The user sees output progressively |
| **Message Batches API** | Non-urgent bulk workloads (nightly reports, large document sets) | 24-hour window, **50% cost reduction** |

#### Synchronous and streaming

Now let's time both. The same request runs once synchronously and once as a stream. For the stream we record when the first piece of text arrives, which is when a user would start reading.

**Sync vs stream:** run the same summary request both ways and compare the wait for the full answer with the wait for the first text.

In [ ]:
prompt = "Summarize the termination rights and notice periods in this contract.\n\n" + contract

# Synchronous: nothing is shown until the whole answer is ready
start = time.time()
r = client.messages.create(model=SONNET, max_tokens=600, messages=[{"role": "user", "content": prompt}])
print(f"Synchronous: full answer after {time.time() - start:.1f}s\n")

# Streaming: text arrives as it is generated
start, first = time.time(), None
with client.messages.stream(model=SONNET, max_tokens=600, messages=[{"role": "user", "content": prompt}]) as stream:
    for text in stream.text_stream:       # each chunk is a few tokens
        if first is None:
            first = time.time() - start
        print(text, end="", flush=True)
print(f"\n\nStreaming: first text after {first:.1f}s, full answer after {time.time() - start:.1f}s")

**Result:** the total time is about the same, but with streaming the user starts reading after a moment instead of waiting for the whole answer. For anything a person is watching, stream.

#### Message Batches API

Say we want three questions answered about the contract, and nobody is waiting for the answers. Instead of sending them one by one, we submit them as one **batch**, then check back for the results. Batches have a **24-hour processing window** and cost **50% less**.

```text
3 questions ──▶ Batch ──▶ poll status ──▶ fetch results
```

> 💡 For high-volume overnight jobs, Haiku plus the Batches API is the cheapest combination: the cheapest model, the batch discount, and no latency requirement.

Now let's run it. Each request gets a `custom_id` so we can match results to questions, because results can come back in any order. A batch usually finishes in a few minutes, so the cell waits and checks every 10 seconds.

**Batch:** submit 3 contract questions as one batch, wait until it ends, then print each answer.

In [ ]:
questions = {
    "parties":     "Who are the parties to this agreement?",
    "termination": "What are the termination conditions?",
    "law":         "Which law governs this agreement?",
}

batch = client.messages.batches.create(requests=[
    {"custom_id": key,                       # lets us match each result to its question
     "params": {"model": HAIKU, "max_tokens": 300,
                "messages": [{"role": "user", "content": f"Contract:\n{contract}\n\nQuestion: {q}\nAnswer only from the contract."}]}}
    for key, q in questions.items()
])
print("Batch submitted:", batch.id)

while (status := client.messages.batches.retrieve(batch.id)).processing_status != "ended":
    print("  status:", status.processing_status)
    time.sleep(10)

for item in client.messages.batches.results(batch.id):
    if item.result.type == "succeeded":
        print(f"\n[{item.custom_id}]", item.result.message.content[0].text)
    else:
        print(f"\n[{item.custom_id}] request {item.result.type}")

**Result:** all three answers came back from one background job. You paid half price and gave up speed. That is the right trade for a nightly report and the wrong one for a chat window.

## M1 Checkpoint

Key facts to remember from this module:

| Concept | One-line takeaway |
|---|---|
| **Tokens** | 1 English word is about 1.3 tokens. Code, JSON and non-English text cost more |
| **Context window** | Input and output share it, so always budget both: `input_tokens + max_tokens ≤ window` |
| **Fill order** | Stable content first, dynamic content last. Never put a timestamp at the front of the system prompt |
| **Sampling** | Low temperature for extraction, high for creative work. Use `temperature` **or** `top_p`, not both |
| **Non-determinism** | `temperature=0` is not fully deterministic. Test properties of the answer, not exact strings |
| **Model choice** | Start with Sonnet. Move up or down based on eval results, not intuition |
| **Alias vs pinned ID** | Alias for development. Pin the ID once you have a validated eval suite |
| **Extended thinking** | Thinking tokens are output tokens. Enable it only when basic mode fails your eval |
| **Prompting modes** | Zero-shot, few-shot, multi-turn. Add only as many examples as the task needs, and cover the hard cases |
| **SDK vs REST** | Use the SDK in production (retries, streaming, types). Raw REST only when the SDK is unavailable |
| **Sync, stream, batch** | Pipelines, user-facing output, and bulk non-urgent work (24 hours, 50% cheaper) |